# Bin-Picking Perception: Baseline → Fine-tune → Evaluate → Error Analysis

Run this top to bottom. See `README.md` in this folder for the full plan and scoping notes.

**Recommended:** upload this notebook to Google Colab (Runtime → Change runtime type → GPU)
for the fine-tune step — it's slow on CPU. Baseline inference and error analysis are fine on CPU.

If running in Colab, also upload (or Google-Drive-mount) the `data/` folder from this project
so the paths below resolve.

## Step 0 — Setup

In [ ]:
# Uncomment the line below if running in Colab (or any fresh environment without these installed)
# !pip install -q ultralytics opencv-python

from pathlib import Path
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".")  # adjust if your working directory differs
RAW_IMAGES_DIR = PROJECT_ROOT / "data" / "raw_images"
DATASET_DIR = PROJECT_ROOT / "data" / "dataset"           # Roboflow export lands here
DATA_YAML = DATASET_DIR / "data.yaml"                      # created by Roboflow export
BASELINE_OUT = PROJECT_ROOT / "results" / "baseline_predictions"
FINETUNED_OUT = PROJECT_ROOT / "results" / "finetuned_predictions"

print("Raw images found:", len(list(RAW_IMAGES_DIR.glob("*.*"))))
print("data.yaml exists:", DATA_YAML.exists(), "-> OK to skip to Step 2 if True, else do Step 1 only")

## Step 1 — Baseline: pretrained model on your RAW (unlabeled) images

This uses a YOLO model pretrained on COCO — it has never seen your objects. The point is to get
an honest "before" picture: does it detect your objects at all, and if so, as what class?
Most small/custom objects (pens, erasers, small tools) are NOT COCO classes, so expect it to
miss them or mislabel them as something COCO-ish. Write down what you actually observe — don't
paper over a bad baseline, that contrast IS the result.

In [ ]:
BASELINE_OUT.mkdir(parents=True, exist_ok=True)

baseline_model = YOLO("yolov8n.pt")  # auto-downloads pretrained COCO weights on first run

baseline_results = baseline_model.predict(
    source=str(RAW_IMAGES_DIR),
    conf=0.25,
    save=True,
    project=str(BASELINE_OUT.parent),
    name=BASELINE_OUT.name,
    exist_ok=True,
)

# Quick tally: what classes did it think it saw, and how often?
from collections import Counter
class_counts = Counter()
for r in baseline_results:
    for c in r.boxes.cls.tolist():
        class_counts[baseline_model.names[int(c)]] += 1
print("Baseline detections by (COCO) class:", dict(class_counts))
print(f"Annotated images saved to: {BASELINE_OUT}")

**Write down now (while it's fresh):** did it detect your objects at all? If so, as what class?
How many images had zero detections? This becomes the "Baseline" line in the README results table.

## Step 2 — Fine-tune on your labeled dataset

Prerequisite: you've annotated `data/raw_images` in Roboflow and exported in **YOLOv8** format
into `data/dataset/`, so that `data/dataset/data.yaml` exists and points at `train/`, `valid/`,
`test/` subfolders. If `data.yaml` doesn't exist yet, stop here and go annotate first.

In [ ]:
assert DATA_YAML.exists(), f"No data.yaml at {DATA_YAML} — export your Roboflow dataset there first."

finetune_model = YOLO("yolov8n.pt")  # start from the same pretrained weights, then adapt

train_results = finetune_model.train(
    data=str(DATA_YAML),
    epochs=100,       # nano model + small dataset trains fast; lower if you're short on time
    imgsz=640,
    patience=20,      # early-stop if val performance plateaus
    project=str(PROJECT_ROOT / "runs"),
    name="finetune",
    exist_ok=True,
)

## Step 3 — Evaluate on the held-out TEST split

Important: evaluate on `test`, not `valid` — `valid` was used (indirectly) during training for
early stopping / checkpoint selection, so `test` is your honest, never-seen-during-training number.

In [ ]:
# Grab the best checkpoint from training
best_weights = PROJECT_ROOT / "runs" / "finetune" / "weights" / "best.pt"
eval_model = YOLO(str(best_weights))

metrics = eval_model.val(data=str(DATA_YAML), split="test")

print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print(f"mAP@0.5:    {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.3f}")
# Copy these numbers into the README Results section — don't round favorably, don't estimate.

## Step 4 — Error analysis

Run predictions on the test split and actually look at the failures. Pick 5–8 concrete examples:
missed objects, wrong class, low-confidence correct detections, duplicate boxes. Save them —
these become your "Failure modes observed" bullets in the README, with evidence to back each one up.

In [ ]:
FINETUNED_OUT.mkdir(parents=True, exist_ok=True)

test_images_dir = DATASET_DIR / "test" / "images"
test_results = eval_model.predict(
    source=str(test_images_dir),
    conf=0.25,
    save=True,
    project=str(FINETUNED_OUT.parent),
    name=FINETUNED_OUT.name,
    exist_ok=True,
)

# Flag the weakest predictions by confidence, as a starting point for manual review
low_conf = []
for r in test_results:
    confs = r.boxes.conf.tolist() if len(r.boxes) else [0.0]
    low_conf.append((Path(r.path).name, min(confs), len(r.boxes)))

low_conf.sort(key=lambda x: x[1])
print("Lowest-confidence test images (inspect these first):")
for name, conf, n_boxes in low_conf[:10]:
    print(f"  {name}: min_conf={conf:.2f}, num_boxes={n_boxes}")

## Step 5 — Close the loop

Go back to `README.md` in this folder and fill in the **Results** section with the real numbers
and real failure examples from above. Be specific and be honest — a correctly-diagnosed mediocre
result is a better interview story than a vague claim of success.